In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split, cross_val_score
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.feature_selection import SelectKBest, chi2
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import train_test_split
import torch
from transformers import BertTokenizer, BertModel

In [ ]:
# Load dataset
df = pd.read_csv("/content/drive/MyDrive/base_tjsp.csv")

# Dataset dimensions
shape_data = df.shape
print(f"The dataset has {shape_data[0]} rows and {shape_data[1]} attributes.")
df.head(3)

In [ ]:
# Class distribution: 'Cybercrime' vs 'Other'
class_distribution = df['classificacao_level0'].value_counts(normalize=True)

for label, proportion in class_distribution.items():
    print(f"Class {label}: {proportion * 100:.2f}%")

In [ ]:
# Concatenate ruling summary (ementa) and full ruling text (julgado)
df_model = df.copy()
df_model["full_text"] = df["ementa"] + " " + df["julgado"]

In [ ]:
# Extract judgment year from date column
df_model['judgment_year'] = df_model['data_julgamento'].str.extract(r'(\d{4})')

# Assign time period based on judgment year
conditions = [
    df_model['judgment_year'].isin(['2010', '2011', '2012', '2013', '2014', '2015']),
    df_model['judgment_year'].isin(['2016', '2017', '2018', '2019', '2020']),
    df_model['judgment_year'].isin(['2021', '2022', '2023', '2024'])
]
choices = ['2010-2015', '2016-2020', 'after 2020']
df_model['period'] = np.select(conditions, choices, default='before 2010')

In [ ]:
lemmatizer = nltk.stem.WordNetLemmatizer()
stopwords_pt = set(stopwords.words('portuguese'))

# Build pattern lists for rapporteur names, cities, and adjudicating bodies to be removed from text
rapporteur_names = "|".join(df_model['relator'].dropna().str.lower().unique())
city_names       = "|".join(df_model['comarca'].dropna().str.lower().unique())
body_names       = "|".join(df_model['orgao_julgador'].dropna().str.lower().unique())


# Text preprocessing function
def preprocess_text(text):
    if pd.isnull(text):
        return ""

    text = text.lower()
    text = re.sub(r".*?(acórdão)", " ", text)

    # Remove headers, signatures, names, dates, and other patterns
    substitutions = [
        r"poder judiciário\s+tribunal de justiça (do estado de são paulo|do estado)?",
        r"tribunal de justiça\s+poder judiciário",
        r"assinatura eletrônica",
        r"\n",
        rapporteur_names,
        city_names,
        body_names,
        r"\b\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}\b",
        r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.(com|com\.br)\b",
        r"https?://\S+|www\.\S+",
        r"[ºª°]",
        r"[0-9]",
        r"\b[A-Za-z]\b",
    ]

    for pattern in substitutions:
        text = re.sub(pattern, " ", text)

    # Remove punctuation and extra whitespace
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    text = word_tokenize(text)

    # Remove stopwords
    text = [word for word in text if word not in stopwords_pt]

    # Lemmatize
    text = [lemmatizer.lemmatize(word) for word in text]

    return text

# Apply preprocessing
df_model['processed_text'] = df_model['full_text'].apply(preprocess_text)

In [ ]:
# Initialize BERTimbau tokenizer and model (Portuguese)
model_name = 'neuralmind/bert-base-portuguese-cased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def generate_bertimbau_embeddings(texts, batch_size=16):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        # Tokenize with truncation, automatic padding, and max length
        inputs = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        # Average token embeddings from the last hidden layer
        batch_embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

# Convert token lists to strings before generating embeddings
texts = [' '.join(tokens) for tokens in df_model['processed_text']]

X_bert = generate_bertimbau_embeddings(texts)

In [ ]:
# One-Hot Encoding for categorical columns
encoder = OneHotEncoder(handle_unknown='ignore')
X_cat = encoder.fit_transform(df_model[['classe', 'assunto', 'topic', 'period']])

# Combine dense BERT embeddings with sparse categorical features
from scipy.sparse import csr_matrix
X_bert_sparse = csr_matrix(X_bert)  # convert to sparse for compatibility
X_combined = hstack([X_bert_sparse, X_cat])

# Define features and target
X = X_combined
y = df_model['classificacao_level0']

# Summary
print(f"Combined feature matrix shape: {X_combined.shape}")

In [ ]:
# Stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    #'NaiveBayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (Linear)': SVC(kernel='linear', probability=True, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42),
    'LightGBM': lgb.LGBMClassifier(random_state=42, verbose=-1)
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('clf', model)
    ])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    mean_score = np.mean(scores)
    std_score  = np.std(scores)

    results.append({
        'model': name,
        'f1_mean': mean_score,
        'f1_std': std_score
    })

    print(f"{name} → Mean F1: {mean_score:.4f} | Std: {std_score:.4f}")

In [ ]:
# Logistic Regression
# Train-test split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Train final model
final_model = LogisticRegression(max_iter=1000, random_state=42)
final_model.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = final_model.predict(X_test_final)
y_prob_final = final_model.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print("\nFinal Evaluation - Logistic Regression (no feature selection):")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",  accuracy_score(y_test_final, y_pred_final))
print("Precision:", precision_score(y_test_final, y_pred_final))
print("Recall:",    recall_score(y_test_final, y_pred_final))
print("F1 Score:",  f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=final_model.classes_)
disp.plot(cmap=cmap, values_format=".1%")
plt.title("Logistic Regression - BERTimbau")
plt.show()

In [ ]:
# Random Forest
# Train-test split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Train final model
final_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_model.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = final_model.predict(X_test_final)
y_prob_final = final_model.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print("\nFinal Evaluation - Random Forest (no feature selection):")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",  accuracy_score(y_test_final, y_pred_final))
print("Precision:", precision_score(y_test_final, y_pred_final))
print("Recall:",    recall_score(y_test_final, y_pred_final))
print("F1 Score:",  f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=final_model.classes_)
disp.plot(cmap=cmap, values_format=".1%")
plt.title("Random Forest - BERTimbau")
plt.show()


In [ ]:
# Naive Bayes
# Train-test split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Train final model
final_model = GaussianNB()
final_model.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = final_model.predict(X_test_final)
y_prob_final = final_model.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print("\nFinal Evaluation - Naive Bayes (no feature selection):")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",  accuracy_score(y_test_final, y_pred_final))
print("Precision:", precision_score(y_test_final, y_pred_final))
print("Recall:",    recall_score(y_test_final, y_pred_final))
print("F1 Score:",  f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=final_model.classes_)
disp.plot(cmap=cmap, values_format=".1%")
plt.title("Naive Bayes - BERTimbau")
plt.show()

In [ ]:
# SVM
# Train-test split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Train final model
final_model = SVC(kernel='linear', probability=True, random_state=42)
final_model.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = final_model.predict(X_test_final)
y_prob_final = final_model.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print("\nFinal Evaluation - SVM (no feature selection):")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",  accuracy_score(y_test_final, y_pred_final))
print("Precision:", precision_score(y_test_final, y_pred_final))
print("Recall:",    recall_score(y_test_final, y_pred_final))
print("F1 Score:",  f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=final_model.classes_)
disp.plot(cmap=cmap, values_format=".1%")
plt.title("SVM - BERTimbau")
plt.show()

In [ ]:
# XGBoost
# Train-test split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Train final model
final_model = XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)
final_model.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = final_model.predict(X_test_final)
y_prob_final = final_model.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print("\nFinal Evaluation - XGBoost (no feature selection):")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",  accuracy_score(y_test_final, y_pred_final))
print("Precision:", precision_score(y_test_final, y_pred_final))
print("Recall:",    recall_score(y_test_final, y_pred_final))
print("F1 Score:",  f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=final_model.classes_)
disp.plot(cmap=cmap, values_format=".1%")
plt.title("XGBoost - BERTimbau")
plt.show()

In [ ]:
# LGBM
# Train-test split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Train final model
final_model = lgb.LGBMClassifier(random_state=42, verbose=-1)
final_model.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = final_model.predict(X_test_final)
y_prob_final = final_model.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print("\nFinal Evaluation - LGBM (no feature selection):")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",  accuracy_score(y_test_final, y_pred_final))
print("Precision:", precision_score(y_test_final, y_pred_final))
print("Recall:",    recall_score(y_test_final, y_pred_final))
print("F1 Score:",  f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=final_model.classes_)
disp.plot(cmap=cmap, values_format=".1%")
plt.title("LGBM - BERTimbau")
plt.show()